In [1]:
import numpy as np
from sympy.combinatorics import Permutation

# Quantum interpreter

An implementation of a quantum interpreter based on Robert Smith's [tutorial](https://www.stylewarning.com/posts/quantum-interpreter/).

## Quantum states

A single qubit has two classical states: $|0\rangle$ and $|1\rangle$. Two qubits have four classical states: $|00\rangle$, $|01\rangle$, $|10\rangle$ and $|11\rangle$, and so on. The rightmost qubit in $|x..x\rangle$ is the least significant qubit as with ordinary binary numbers.

Unlike classic bits, qubits can be in a superposition ofs classical states, that is, the state of a qubit is a linear combination ("weighted average") of the classical states. For example, for two qubits the linear combination might be:

| A | B | AB | Amplitude | Probability |
|-|-|-|-|-|
| $\|0\rangle$ | $\|0\rangle$ | $\|00\rangle$ | $1/\sqrt(2)$ | 50% |
| $\|0\rangle$ | $\|1\rangle$ | $\|01\rangle$ | 0 | 0% |
| $\|1\rangle$ | $\|0\rangle$ | $\|10\rangle$ | 0 | 0% |
| $\|1\rangle$ | $\|1\rangle$ | $\|11\rangle$ | $1/\sqrt(2)$ | 50% |

The "weights" in the "weighted average" are the complex probability amplitudes of the corresponding classical state. The squares of the probability amplitudes have to sum to 1.

A quantum state consists of the probability amplitudes of the classical states.
The first entry in the quantum state vector corresponds to $|...000\rangle$ and
the last entry corresonds to $|...111\rangle$. We can use the binary number that
corresponds a classical state to index into the quantum state vector and to find
its probability amplitude.

In [2]:
class State:
    """A quantum state."""

    def __init__(self, n: int):
        """Create a quantum state for n qubits."""
        self.numberOfQubits = n
        self.collapse(0)

    def collapse(self, i : int):
        """Collapse the quantum state to the classical state i.
        
        Arguments:
            i       The index of the classical state.
        """
        self.state = np.zeros(shape=2**self.numberOfQubits, dtype=np.complex64)
        self.state[i] = 1+0j

    def __repr__(self):
        n = self.numberOfQubits
        result = f"State of {n} qubits with amplitudes and probabilities:\n"
        for i, entry in enumerate(self.state):
            result += f"|{i:0{n}b}>{entry:20.6}{abs(entry**2):20.6}\n"
        return result

In [3]:
State(1)

State of 1 qubits with amplitudes and probabilities:
|0>              (1+0j)                 1.0
|1>                  0j                 0.0

In [4]:
State(2)

State of 2 qubits with amplitudes and probabilities:
|00>              (1+0j)                 1.0
|01>                  0j                 0.0
|10>                  0j                 0.0
|11>                  0j                 0.0

## Quantum operators

Quantum operators are represented as unitary matrices that modify a quantum state.

In [5]:
type Matrix = list[list[np.complex64]]

class Operator:
    """A quantum operator."""

    def __init__(self, name: str, matrix: Matrix):
        """Create a quantum operator from a unitary matrix.
        
        Arguments:
           gate         The gate matrix.
           name         An optional name for the operator.

        Asserts the gate matrix is unitary.
        """
        self.name = name
        self.matrix = np.array(matrix)
        self.numberOfQubits = int(np.log2(self._numberOfStates()))
        self._assertUnitary()

    def __repr__(self):
        return f"Operator {self.name}:\n{self.matrix}"

    def __eq__(self, other):
        return (
            isinstance(other, Operator) and
            self.name == other.name and
            self.numberOfQubits == other.numberOfQubits and
            np.allclose(self.matrix, other.matrix, atol=0.001))

    def _assertUnitary(self):
        """Asserts the operator is a unitary matrix."""
        u = self.matrix
        i = np.identity(self._numberOfStates())
        assert(np.allclose(u @ u.conjugate().transpose(), i, atol=0.001))

    def _numberOfStates(self) -> int:
        """Asserts the operator is a unitary matrix."""
        return self.matrix.shape[0]

In [6]:
# Hadamard gate that puts a single qubit into superposition.
SQRT_ONE_HALF = np.sqrt(0.5, dtype=np.complex64)
H = Operator("H", [
    [SQRT_ONE_HALF, SQRT_ONE_HALF],     # |0> -> |+>
    [SQRT_ONE_HALF, -SQRT_ONE_HALF],    # |1> -> |->
])

# Controlled NOT on two qubits. The least qubit is the control qubit.
# If the control qubit is (0), the target is not modified.
# If the control qubit is (1), the target is inverted.
CNOT = Operator("CNOT", [
    [1, 0, 0, 0],   # |00> -> |00>
    [0, 1, 0, 0],   # |01> -> |01>
    [0, 0, 0, 1],   # |10> -> |11>
    [0, 0, 1, 0],   # |11> -> |10>
])

Quantum states are represented as vectors, and quantum operators are represented as matrices. Thus transforming a quantum state or applying an operation to it is the same as matrix-vector multiplication.

In [7]:
def applyToAllQubits(state: State, operator: Operator):
    """Apply a quantum operator to all qubits of a quantum state.
    
    Arguments:
        state           The quantum state.
        operator        The operator to apply.
    """
    # A quantum operator is a unitary matrix. Applying the
    # operator is the same as matrix-vector multiplication.
    result = State(state.numberOfQubits)
    result.state = operator.matrix.dot(state.state)
    return result

In [8]:
s = State(1)
applyToAllQubits(s, H)

State of 1 qubits with amplitudes and probabilities:
|0>       (0.707107+0j)                 0.5
|1>       (0.707107+0j)                 0.5

Multiple quantum operators can be combined into a single operator with matrix multiplication.

In [9]:
from functools import reduce

def combine(name: str, operators: list[Operator]):
    """Combine quantum operators into a single operator.
    
    Arguments:
        name            The name of the new operator.
        operators        A list of quantum operators.
    """
    [first, *rest] = operators
    return reduce(lambda a, b: Operator(name, np.matmul(a.matrix, b.matrix)),
                  rest, first)

In [10]:
HH = combine("HH", [H, H])
HH

Operator HH:
[[ 9.99999940e-01+0.j -1.26880515e-08+0.j]
 [-1.26880515e-08+0.j  9.99999940e-01+0.j]]

In [11]:
applyToAllQubits(s, HH)

State of 1 qubits with amplitudes and probabilities:
|0>              (1+0j)                 1.0
|1>   (-1.26881e-08+0j)         1.60987e-16

## Measurements

A measurement on the quantum state vector collapses the quantum state into a classical state. We view the quantum state as a discrete probability distribution and we sample from it.

In [12]:
type Counts = np.ndarray[tuple[int], np.dtype[np.int64]]
type DiscreteDistribution = np.ndarray[tuple[int], np.dtype[np.float64]]

def probabilities(state: State) -> DiscreteDistribution:
    """Return the discrete probability distribution corresponding
    to the quantum state.
    
    Arguments:
        state   The quantum state.
    """
    return np.abs(np.square(state.state))

def sample(state: State, n: int = 1) -> Counts:
    """Sample n times from the probability distribution of the quantum
    state.
    
    Arguments:
        state   The quantum state.
        n       The number of times to sample from the
                probability distribution of the quantum state.
    """
    probs = probabilities(state)
    states = list(range(len(probs)))

    samples = np.random.choice(states, n, p=probs)
    return np.unique_counts(samples).counts

def measure(state: State, n: int = 1) -> State:
    """Measure and collapse the quantum state.
    
    Arguments:
        state   The quantum state.
        n       The number of times to sample from the
                probability distribution of the quantum state.
    """
    counts = sample(state, n)
    result = State(state.numberOfQubits)
    result.collapse(np.argmax(counts))
    return result

In [13]:
s = State(1)
s = applyToAllQubits(s, H)
probabilities(s)

array([0.49999997, 0.49999997], dtype=float32)

In [14]:
sample(s, 100)

array([47, 53])

In [15]:
s = measure(s, 100)
s

State of 1 qubits with amplitudes and probabilities:
|0>              (1+0j)                 1.0
|1>                  0j                 0.0

## Quantum gates

We represent the $2^n$ classical states for $n$ qubits as a $2^n$-dimensional vector. Therefore a quantum gate operating on $n$ qubits is a $2^n$-dimensional unitary matrix.

Simple quantum gates are I (Identity) and X (NOT).

In [16]:
# Identity.
I = Operator("I", [
    [1, 0],     # |0> -> |0>
    [0, 1],     # |1> -> |1>
])

# X gate (NOT gate).
X = Operator("X", [
    [0, 1],     # |0> -> |1>
    [1, 0],     # |1> -> |0>
])

### Gates on adjacent qubits

A gate on a single qubit in a quantum state modifies only that qubit. That is, we apply the gate $g$ to qubit $i$ and we apply the identity $I$ to the other qubits:

\begin{equation}
\operatorname{lift}(g, i, n) :=
\underbrace{\mathsf{I} \otimes \mathsf{I} \otimes \cdots}_{n-i-1\text{ factors}}
\otimes g \otimes
\underbrace{\cdots \otimes \mathsf{I}}_{i\text{ factors}},
\end{equation}

([Equation (2) in the tutorial](https://www.stylewarning.com/posts/quantum-interpreter/))

And we can generalize that if the gate applies to $k$ adjacent qubits:

\begin{equation}
\operatorname{lift}(g, i, n) := \underbrace{\mathsf{I} \otimes \mathsf{I} \otimes \cdots}_{n-i-k\text{ factors}}
\otimes g \otimes
\underbrace{\cdots \otimes \mathsf{I}}_{i\text{ factors}}.
\end{equation}

([Equation (3) in the tutorial](https://www.stylewarning.com/posts/quantum-interpreter/))

$\otimes$ is the Kronecker-Product of two matrices:

\begin{aligned}
\begin{pmatrix}
a_{11} & a_{12} \\
a_{21} & a_{22}
\end{pmatrix}
\otimes
\begin{pmatrix}
b_{11} & b_{12} \\
b_{21} & b_{22}
\end{pmatrix}
&=
\begin{pmatrix}
a_{11} \begin{pmatrix}
b_{11} & b_{12} \\
b_{21} & b_{22}
\end{pmatrix} &
a_{12} \begin{pmatrix}
b_{11} & b_{12} \\
b_{21} & b_{22}
\end{pmatrix}\\
a_{21} \begin{pmatrix}
b_{11} & b_{12} \\
b_{21} & b_{22}
\end{pmatrix} &
a_{22} \begin{pmatrix}
b_{11} & b_{12} \\
b_{21} & b_{22}
\end{pmatrix}
\end{pmatrix} \\
&=
\begin{pmatrix}
a_{11} b_{11} &
a_{11} b_{12} &
a_{12} b_{11} &
a_{12} b_{12} \\
a_{11} b_{21} &
a_{11} b_{22} &
a_{12} b_{21} &
a_{12} b_{22} \\
a_{21} b_{11} &
a_{21} b_{12} &
a_{22} b_{11} &
a_{22} b_{12} \\
a_{21} b_{21} &
a_{21} b_{22} &
a_{22} b_{21} &
a_{22} b_{22}
\end{pmatrix}
\end{aligned}

In [17]:
from functools import reduce

def kroneckerExp(u: Operator, n: int) -> Operator:
    """Apply operator u n-times to itself with the Kronecker-product.
    
    Arguments:
        u       The operator for which to compute the Kronecker-exponential.
        n       The exponent of the Kronecker-exponential.
    """
    name = f"{u.name}^{n}"
    if n < 1:
        return Operator(name, np.ones((1,)))
    return Operator(name, reduce(np.kron, [u.matrix] * (n-1), u.matrix))

In [18]:
kroneckerExp(I, 1)

Operator I^1:
[[1 0]
 [0 1]]

In [19]:
kroneckerExp(I, 2)

Operator I^2:
[[1 0 0 0]
 [0 1 0 0]
 [0 0 1 0]
 [0 0 0 1]]

In [20]:
kroneckerExp(I, 3)

Operator I^3:
[[1 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0]
 [0 0 0 1 0 0 0 0]
 [0 0 0 0 1 0 0 0]
 [0 0 0 0 0 1 0 0]
 [0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 1]]

In [21]:
def lift(u: Operator, i: int, n: int) -> Operator:
    """Lift a quantum gate to qubit i in a quantum state of n qubits.
    
    Arguments:
        u       The quantum gate to lift.
        i       The qubit to which the gate applies.
        n       The number of qubits in the quantum state.
    """
    left = kroneckerExp(I, n - i - u.numberOfQubits)
    right = kroneckerExp(I, i)
    return Operator(f"{u.name} on qubit {i} of {n}",
                    np.kron(left.matrix, np.kron(u.matrix, right.matrix)))

def applyToAdjacentQubits(state: State, u: Operator, qubit: int) -> State:
    """Apply operator to adjacent qubits in state.
    
    Arguments:
        state       The quantum state.
        u           The operator to apply to the qubit in the state.
        qubit       The least-significant qubit to which the operator applies.
    """
    operator = lift(u, qubit, state.numberOfQubits)
    return applyToAllQubits(state, operator)

In [22]:
# Apply X to qubit 0 of a 4-qubit machine, that is, swap |...0> and |...1>.
lift(X, 0, 4)

Operator X on qubit 0 of 4:
[[0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]]

In [23]:
state = State(4)
state.state = np.stack([1.0, 0.0] * 2 ** (state.numberOfQubits-1))
state

State of 4 qubits with amplitudes and probabilities:
|0000>                 1.0                 1.0
|0001>                 0.0                 0.0
|0010>                 1.0                 1.0
|0011>                 0.0                 0.0
|0100>                 1.0                 1.0
|0101>                 0.0                 0.0
|0110>                 1.0                 1.0
|0111>                 0.0                 0.0
|1000>                 1.0                 1.0
|1001>                 0.0                 0.0
|1010>                 1.0                 1.0
|1011>                 0.0                 0.0
|1100>                 1.0                 1.0
|1101>                 0.0                 0.0
|1110>                 1.0                 1.0
|1111>                 0.0                 0.0

In [24]:
applyToAdjacentQubits(state, X, 0)

State of 4 qubits with amplitudes and probabilities:
|0000>                 0.0                 0.0
|0001>                 1.0                 1.0
|0010>                 0.0                 0.0
|0011>                 1.0                 1.0
|0100>                 0.0                 0.0
|0101>                 1.0                 1.0
|0110>                 0.0                 0.0
|0111>                 1.0                 1.0
|1000>                 0.0                 0.0
|1001>                 1.0                 1.0
|1010>                 0.0                 0.0
|1011>                 1.0                 1.0
|1100>                 0.0                 0.0
|1101>                 1.0                 1.0
|1110>                 0.0                 0.0
|1111>                 1.0                 1.0

### Gates on multiple non-adjacent qubits

The idea is to move non-adjacent qubits next to each other, to apply the operator to the now adjacent qubits, and then to move the qubits back to their original positions.

We move qubits by consecutively swapping two adjacent qubits.

In [25]:
# Swap two qubits.
SWAP = Operator("SWAP", [
    [1, 0, 0, 0],   # |00> -> |00>
    [0, 0, 1, 0],   # |01> -> |10>
    [0, 1, 0, 0],   # |10> -> |01>
    [0, 0, 0, 1],   # |11> -> |11>
])

Gates in a quantum circuit specify the qubits to which they apply. We interpret this qubit list as a permutation and we transpose the qubits in the state so that the least significant qubits are the qubits for the gate.

In [26]:
import itertools
from typing import Any

def flatten(l : list[list[Any]]) -> list[Any]:
    """Flatten a list of lists into a list.
    
    Arguments:
        l       The list of lists.
    """
    return list(itertools.chain.from_iterable(l))

def expandTransposition(transposition: tuple[int, int]) -> list[int]:
    """Expand a transposition into adjacent transpositions.
    
    Arguments:
        transposition       A transposition of the form (from, to).
    """
    (f, t) = transposition
    left = range(f, t-1)
    return flatten([left, [t-1], reversed(left)])

def adjacentTranspositions(permutation: list[int]) -> list[int]:
    """Converts a permutation into a series of adjacent transpositions.
    
    Arguments:
        permutation         A permutation list.
    """
    transpositions = Permutation(permutation).transpositions()
    expanded = map(expandTransposition, transpositions)
    return list(itertools.chain.from_iterable(expanded))

In [27]:
adjacentTranspositions([3, 4, 0, 2, 1])

[0, 1, 0, 0, 1, 2, 1, 0, 1, 2, 3, 2, 1]

In [28]:
def swap(l: list[Any], i: int):
    temp = l[i]
    l[i] = l[i+1]
    l[i+1] = temp

value = [0, 1, 2, 3, 4]
permutation = [3, 4, 0, 2, 1]
for i in adjacentTranspositions(permutation):
    swap(value, i)
assert(value == permutation)

In [29]:
type QubitList = list[int]

def qubitsToPermutation(qubits: QubitList, n: int) -> list[int]:
    """Expand a subset of qubits to a permutation of n qubits.
    
    Arguments:
        qubits      A list of qubit indices.
        n           The total number of qubits.
    """
    # Reverse the order of the selected qubits because the first item in
    # the list is the least-significant one, that is, the rightmost qubit.
    return list(reversed(qubits)) + [i for i in range(n) if i not in qubits]

In [30]:
qubitsToPermutation([3, 4], 5)

[4, 3, 0, 1, 2]

In [31]:
def applyToNonAdjacentQubits(
        state: State,
        u: Operator,
        qubits: QubitList) -> State:
    """Apply operator u to qubits in state.
    
    Arguments:
        state       The quantum state.
        u           The operator to apply to the qubits in the state.
        qubits      A list of qubits to which the operator applies.
    """
    n = state.numberOfQubits
    transpositions = adjacentTranspositions(qubitsToPermutation(qubits, n))

    # Swap the selected qubits into place.
    swap = [lift(SWAP, t, state.numberOfQubits)
            for t in transpositions]

    # The operator we're interested in.
    operator = [lift(u, 0, state.numberOfQubits)]
    
    # Restore the original configuration.
    restore = [lift(SWAP, t, state.numberOfQubits)
               for t in reversed(transpositions)]

    return applyToAllQubits(state, combine(u.name, swap + operator + restore))

In [32]:
state = State(4)
state.collapse(2)
state

State of 4 qubits with amplitudes and probabilities:
|0000>                  0j                 0.0
|0001>                  0j                 0.0
|0010>              (1+0j)                 1.0
|0011>                  0j                 0.0
|0100>                  0j                 0.0
|0101>                  0j                 0.0
|0110>                  0j                 0.0
|0111>                  0j                 0.0
|1000>                  0j                 0.0
|1001>                  0j                 0.0
|1010>                  0j                 0.0
|1011>                  0j                 0.0
|1100>                  0j                 0.0
|1101>                  0j                 0.0
|1110>                  0j                 0.0
|1111>                  0j                 0.0

In [33]:
applyToNonAdjacentQubits(state, SWAP, [1, 3])

State of 4 qubits with amplitudes and probabilities:
|0000>                  0j                 0.0
|0001>                  0j                 0.0
|0010>                  0j                 0.0
|0011>                  0j                 0.0
|0100>                  0j                 0.0
|0101>                  0j                 0.0
|0110>                  0j                 0.0
|0111>                  0j                 0.0
|1000>              (1+0j)                 1.0
|1001>                  0j                 0.0
|1010>                  0j                 0.0
|1011>                  0j                 0.0
|1100>                  0j                 0.0
|1101>                  0j                 0.0
|1110>                  0j                 0.0
|1111>                  0j                 0.0

We can finally apply gates any qubits in a quantum state if they are compatible.

In [34]:
def applyOperator(
        state: State,
        u: Operator,
        qubits: QubitList) -> State:
    """Apply operator u to qubits in state.
    
    Arguments:
        state       The quantum state.
        u           The operator to apply to the qubits in the state.
        qubits      A list of qubits to which the operator applies.
    """
    assert(len(qubits) == u.numberOfQubits)
    assert(len(qubits) <= state.numberOfQubits)

    match qubits:
        case [qubit]:
            return applyToAdjacentQubits(state, u, qubit)
        case qubits:
            return applyToNonAdjacentQubits(state, u, qubits)

## Quantum circuits

The fundamental operations of a quantum computer are gates and measurements. Gates are unitary matrices that modify the quantum state. Measurements observe the current quantum state and collapse it into a classical state.

You can think of gates and measurements are basic instructions in a quantum computing assembly language. Another interpretation sees them as basic circuit elements in a quantum circuit similar to logic gates NOT, AND, OR in digital circuits.

In [35]:
class Gate:
    """A quantum gate."""

    # Enable pattern matching on object attributes.
    __match_args__ = ('operator', 'qubits')

    def __init__(self, u: Operator, qubits : QubitList):
        """Create a quantum gate.
        
        Arguments:
            u           A unitary matrix describing the gate.
            qubits      The indices of the qubits in the quantum
                        state to which the gate applies.
        """
        self.operator = u
        self.qubits = qubits

    def __repr__(self):
        op = self.operator
        return f"Gate {op.name} on qubits {self.qubits}:\n{op.matrix}"

    def __eq__(self, other):
        return (
            isinstance(other, Gate) and
            self.qubits == other.qubits and
            self.operator == other.operator)

class Measure:
    """A quantum measurement."""

    def __init__(self):
        """Create a quantum measurement."""

    def __repr__(self):
        return f"Measure"

    def __eq__(self, other):
        return isinstance(other, Measure)

In [36]:
ExampleBell25 = [
    Gate(H, [2]),
    Gate(CNOT, [2, 5]),
    Measure(),
]
ExampleBell25

[Gate H on qubits [2]:
 [[ 0.70710677+0.j  0.70710677+0.j]
  [ 0.70710677+0.j -0.70710677-0.j]],
 Gate CNOT on qubits [2, 5]:
 [[1 0 0 0]
  [0 1 0 0]
  [0 0 0 1]
  [0 0 1 0]],
 Measure]

In [37]:
class Circuit:
    """A quantum circuit."""

    def __init__(self, n: int):
        """Create an empty quantum circuit of n qubits.
        
        Arguments:
            n       The number of qubits in the circuit.
        """
        self.numberOfQubits = n
        self.operations = []

    def add(self, gate: Operator, qubits : QubitList):
        """Add a gate to the circuit."""
        assert(all(qubit < self.numberOfQubits for qubit in qubits))
        self.operations.append(Gate(gate, qubits))

    def measure(self):
        """Measure and collapse the current quantum state."""
        self.operations.append(Measure())

    def diff(self, other: Circuit):
        """Compare this circuit to another one."""
        assert(self.numberOfQubits == other.numberOfQubits)
        assert(len(self.operations) == len(other.operations))

        for (i, (a, b)) in enumerate(zip(self.operations, other.operations)):
            if a != b:
                print(f"Different operation {i}:")
                print(f"    This circuit: {a}")
                print(f"    That circuit: {b}")

In [38]:
ExampleBell25 = Circuit(6)
ExampleBell25.add(H, [2])
ExampleBell25.add(CNOT, [2, 5])
ExampleBell25.measure()
ExampleBell25.operations

[Gate H on qubits [2]:
 [[ 0.70710677+0.j  0.70710677+0.j]
  [ 0.70710677+0.j -0.70710677-0.j]],
 Gate CNOT on qubits [2, 5]:
 [[1 0 0 0]
  [0 1 0 0]
  [0 0 0 1]
  [0 0 1 0]],
 Measure]

## Quantum interpreter

In [39]:
def run(circuit: Circuit, initialState: State = None) -> State:
    """Run a quantum circuit on an initial state.
    
    Arguments:
        circuit         The quantum circuit.
        initialState    The optional initial state.
                        |0..0> if no state is given.
    """
    state = initialState or State(circuit.numberOfQubits)
    for operation in circuit.operations:
        match operation:
            case Gate(operator, qubits):
                state = applyOperator(state, operator, qubits)
            case Measure():
                state = measure(state)
    return state

## Examples

### Bell state

$\frac{1}{\sqrt{2}}(\ket {00} + \ket {11})$

In [40]:
circuit = Circuit(2)
circuit.add(H, [0])
circuit.add(CNOT, [0, 1])
run(circuit)

State of 2 qubits with amplitudes and probabilities:
|00>       (0.707107+0j)                 0.5
|01>                  0j                 0.0
|10>                  0j                 0.0
|11>       (0.707107+0j)                 0.5

### Greenberger-Horne-Zeilinger state

A generalized Bell state: $\frac{1}{\sqrt{2}}(\ket{0\ldots 000} + \ket{1\ldots 111})$

In [41]:
def ghz(n):
    circuit = Circuit(n)
    circuit.add(H, [0])
    for i in range(n-1):
        circuit.add(CNOT, [i, i+1])
    return circuit

run(ghz(4))

State of 4 qubits with amplitudes and probabilities:
|0000>       (0.707107+0j)                 0.5
|0001>                  0j                 0.0
|0010>                  0j                 0.0
|0011>                  0j                 0.0
|0100>                  0j                 0.0
|0101>                  0j                 0.0
|0110>                  0j                 0.0
|0111>                  0j                 0.0
|1000>                  0j                 0.0
|1001>                  0j                 0.0
|1010>                  0j                 0.0
|1011>                  0j                 0.0
|1100>                  0j                 0.0
|1101>                  0j                 0.0
|1110>                  0j                 0.0
|1111>       (0.707107+0j)                 0.5

### Quantum Fourier transform

In [42]:
def CPHASE(angle: float) -> Operator:
    """Create a CPHASE quantum gate.
    
    Arguments:
        angle       The phase angle in radians.
    """
    cis = np.complex64(np.cos(angle), np.sin(angle))
    return Operator(f"CPHASE({angle})", [
        [1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 1, 0],
        [0, 0, 0, cis]
    ])

In [43]:
def reverseQubits(circuit: Circuit, qubits: QubitList):
    """Reverse qubits in a quantum circuit:
    
            swap(q[0], q[n-1])
            swap(q[1], q[n-2])
            swap(q[2], q[n-3])
            ...

    Arguments:
        circuit         The quantum circuit in which to swap qubits.
                        This function adds SWAP gates to the circuit.
        qubits          The qubits to swap.
    """
    n = len(qubits) // 2
    for (a, b) in zip(qubits[:n], reversed(qubits[n:])):
        circuit.add(SWAP, [a, b])

In [44]:
circuit = Circuit(3)
reverseQubits(circuit, list(range(circuit.numberOfQubits)))
circuit.operations

[Gate SWAP on qubits [0, 2]:
 [[1 0 0 0]
  [0 0 1 0]
  [0 1 0 0]
  [0 0 0 1]]]

In [45]:
def addCPHASE(circuit: Circuit, qubit: int, rest: QubitList):
    """Add CPHASE gates betweeen qubit and the other qubits.
    
    Arguments:
        circuit     The quantum circuit to which to add the QFT.
        qubit       The first qubit.
        rest        The other qubits on which to calculate the QFT.
    """
    n = len(rest)
    for (i, other) in enumerate(rest):
        angle = np.pi / (2 ** (n - i))
        circuit.add(CPHASE(angle), [qubit, other])

In [46]:
circuit = Circuit(3)
addCPHASE(circuit, 0, [1, 2])
circuit.operations

[Gate CPHASE(0.7853981633974483) on qubits [0, 1]:
 [[1.        +0.j         0.        +0.j         0.        +0.j
   0.        +0.j        ]
  [0.        +0.j         1.        +0.j         0.        +0.j
   0.        +0.j        ]
  [0.        +0.j         0.        +0.j         1.        +0.j
   0.        +0.j        ]
  [0.        +0.j         0.        +0.j         0.        +0.j
   0.70710677+0.70710677j]],
 Gate CPHASE(1.5707963267948966) on qubits [0, 2]:
 [[1.00000000e+00+0.j 0.00000000e+00+0.j 0.00000000e+00+0.j
   0.00000000e+00+0.j]
  [0.00000000e+00+0.j 1.00000000e+00+0.j 0.00000000e+00+0.j
   0.00000000e+00+0.j]
  [0.00000000e+00+0.j 0.00000000e+00+0.j 1.00000000e+00+0.j
   0.00000000e+00+0.j]
  [0.00000000e+00+0.j 0.00000000e+00+0.j 0.00000000e+00+0.j
   6.12323426e-17+1.j]]]

In [47]:
def qft(circuit: Circuit, qubits: QubitList):
    """Add a quantum Fourier transform to the circuit.
    
    Arguments:
        circuit     The quantum circuit to which to add the QFT.
        qubits      The qubits on which to calculate the QFT.
    """
    [qubit, *rest] = qubits
    if rest != []:
        qft(circuit, rest)
        addCPHASE(circuit, qubit, rest)
    circuit.add(H, [qubit])
    reverseQubits(circuit, qubits)

In [48]:
circuit = Circuit(3)
qft(circuit, list(range(circuit.numberOfQubits)))
circuit.operations

[Gate H on qubits [2]:
 [[ 0.70710677+0.j  0.70710677+0.j]
  [ 0.70710677+0.j -0.70710677-0.j]],
 Gate CPHASE(1.5707963267948966) on qubits [1, 2]:
 [[1.00000000e+00+0.j 0.00000000e+00+0.j 0.00000000e+00+0.j
   0.00000000e+00+0.j]
  [0.00000000e+00+0.j 1.00000000e+00+0.j 0.00000000e+00+0.j
   0.00000000e+00+0.j]
  [0.00000000e+00+0.j 0.00000000e+00+0.j 1.00000000e+00+0.j
   0.00000000e+00+0.j]
  [0.00000000e+00+0.j 0.00000000e+00+0.j 0.00000000e+00+0.j
   6.12323426e-17+1.j]],
 Gate H on qubits [1]:
 [[ 0.70710677+0.j  0.70710677+0.j]
  [ 0.70710677+0.j -0.70710677-0.j]],
 Gate SWAP on qubits [1, 2]:
 [[1 0 0 0]
  [0 0 1 0]
  [0 1 0 0]
  [0 0 0 1]],
 Gate CPHASE(0.7853981633974483) on qubits [0, 1]:
 [[1.        +0.j         0.        +0.j         0.        +0.j
   0.        +0.j        ]
  [0.        +0.j         1.        +0.j         0.        +0.j
   0.        +0.j        ]
  [0.        +0.j         0.        +0.j         1.        +0.j
   0.        +0.j        ]
  [0.        +0.j 

In [49]:
reference = Circuit(3)
reference.add(H, [2])
reference.add(CPHASE(np.pi/2), [1, 2])
reference.add(H, [1])
reference.add(SWAP, [1, 2])
reference.add(CPHASE(np.pi/4), [0, 1])
reference.add(CPHASE(np.pi/2), [0, 2])
reference.add(H, [0])
reference.add(SWAP, [0, 2])
circuit.diff(reference)

In [50]:
result = run(circuit)
result

State of 3 qubits with amplitudes and probabilities:
|000>       (0.353553+0j)               0.125
|001>       (0.353553+0j)               0.125
|010>       (0.353553+0j)               0.125
|011>       (0.353553+0j)               0.125
|100>       (0.353553+0j)               0.125
|101>       (0.353553+0j)               0.125
|110>       (0.353553+0j)               0.125
|111>       (0.353553+0j)               0.125

In [ ]:
reference = np.fft.fft([1,0,0,0,0,0,0,0], norm="ortho")
assert(np.allclose(result.state, reference))